In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# Attention Visualization\n", "Visualize what each attention head focuses on for a given sentence."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "import torch\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.ticker as ticker\n",
    "import numpy as np\n",
    "from transformer import make_transformer\n",
    "from data import build_dataloaders, tokenize, SOS_IDX, EOS_IDX, PAD_IDX\n",
    "from inference import translate, greedy_decode\n",
    "\n",
    "device = torch.device('cpu')\n",
    "_, _, src_vocab, tgt_vocab = build_dataloaders(batch_size=32)\n",
    "\n",
    "model = make_transformer(\n",
    "    src_vocab_size=len(src_vocab),\n",
    "    tgt_vocab_size=len(tgt_vocab),\n",
    "    d_model=256, h=8, N=3, d_ff=512, dropout=0.0\n",
    ").to(device)\n",
    "\n",
    "model.load_state_dict(torch.load('../checkpoints/best_model.pt', map_location=device))\n",
    "model.eval()\n",
    "print('Model loaded.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def get_attention_weights(sentence, model, src_vocab, tgt_vocab, device):\n",
    "    \"\"\"Run a forward pass and collect attention weights from all heads.\"\"\"\n",
    "    tokens = tokenize(sentence)\n",
    "    src_ids = [SOS_IDX] + src_vocab.encode(tokens) + [EOS_IDX]\n",
    "    src = torch.tensor([src_ids], dtype=torch.long, device=device)\n",
    "    src_mask = (src != PAD_IDX).unsqueeze(1).unsqueeze(2)\n",
    "    src_tokens = ['<sos>'] + tokens + ['<eos>']\n",
    "\n",
    "    with torch.no_grad():\n",
    "        enc_out = model.encode(src, src_mask)\n",
    "        tgt = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)\n",
    "        tgt_tokens = ['<sos>']\n",
    "\n",
    "        for _ in range(50):\n",
    "            tgt_mask = torch.tril(torch.ones(1, 1, tgt.size(1), tgt.size(1))).bool()\n",
    "            dec_out = model.decode(tgt, enc_out, src_mask, tgt_mask)\n",
    "            logits = model.projection(dec_out[:, -1, :])\n",
    "            next_token = logits.argmax(dim=-1).unsqueeze(0)\n",
    "            tgt = torch.cat([tgt, next_token], dim=1)\n",
    "            tgt_tokens.append(tgt_vocab.idx2token.get(next_token.item(), '<unk>'))\n",
    "            if next_token.item() == EOS_IDX:\n",
    "                tgt_tokens.append('<eos>')\n",
    "                break\n",
    "\n",
    "    # Collect cross-attention weights from last decoder layer\n",
    "    attn_weights = model.decoder.layers[-1].cross_attention.attention_weights\n",
    "    return attn_weights.squeeze(0).cpu().numpy(), src_tokens, tgt_tokens\n",
    "\n",
    "\n",
    "def plot_attention_heads(weights, src_tokens, tgt_tokens, sentence):\n",
    "    \"\"\"Plot all 8 attention heads in a grid.\"\"\"\n",
    "    n_heads = weights.shape[0]\n",
    "    fig, axes = plt.subplots(2, 4, figsize=(20, 10))\n",
    "    fig.suptitle(f'Cross-Attention Weights\\n\"{sentence}\"', fontsize=14)\n",
    "\n",
    "    for h, ax in enumerate(axes.flat):\n",
    "        w = weights[h, :len(tgt_tokens), :len(src_tokens)]\n",
    "        im = ax.imshow(w, cmap='Blues', aspect='auto')\n",
    "        ax.set_xticks(range(len(src_tokens)))\n",
    "        ax.set_yticks(range(len(tgt_tokens)))\n",
    "        ax.set_xticklabels(src_tokens, rotation=45, ha='right', fontsize=8)\n",
    "        ax.set_yticklabels(tgt_tokens, fontsize=8)\n",
    "        ax.set_title(f'Head {h+1}')\n",
    "        plt.colorbar(im, ax=ax)\n",
    "\n",
    "    plt.tight_layout()\n",
    "    plt.savefig('../attention_maps.png', dpi=150, bbox_inches='tight')\n",
    "    plt.show()\n",
    "    print('Saved to attention_maps.png')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "sentence = 'ein mann spielt gitarre .'\n",
    "weights, src_tokens, tgt_tokens = get_attention_weights(sentence, model, src_vocab, tgt_vocab, device)\n",
    "print(f'Attention weight shape: {weights.shape}')  # (8 heads, tgt_len, src_len)\n",
    "print(f'Source tokens: {src_tokens}')\n",
    "print(f'Target tokens: {tgt_tokens}')\n",
    "plot_attention_heads(weights, src_tokens, tgt_tokens, sentence)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.14.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}